[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Biswajit1999/daily-astro-notebooks/blob/master/gaia/2026-07-30-hr-diagram-ngc188/notebook.ipynb)

# NGC 188: an old open cluster's HR diagram, and what its turnoff tells us

**Learning goals** — after this notebook you'll be able to:
- Query and quality-cut a real Gaia DR3 sample for a distant (~1.8 kpc), old open cluster.
- Build an absolute-magnitude HR diagram and locate a genuine main-sequence turnoff (a real bend, not just "bluest point").
- Quantitatively compare NGC 188's turnoff to the Hyades and Pleiades on one figure to rank the three clusters by age.

**Background.** NGC 188 is one of the oldest known nearby open clusters, roughly 6-7 billion years old — much older than the Hyades (~700 Myr) or Pleiades (~100 Myr). Because it's older, more of its higher-mass main-sequence stars have already evolved into red giants, so its main-sequence **turnoff** — where the tight main-sequence band bends up and to the right toward the giant branch — sits at a fainter, redder color than younger clusters. Locating that bend precisely is one of the standard ways to estimate a cluster's age from photometry alone.

## 1. Query real Gaia DR3 data for the NGC 188 field

In [ ]:
from astroquery.gaia import Gaia
import numpy as np
import matplotlib.pyplot as plt

# NGC 188 center; it's compact and distant (~1.8 kpc) so a modest cone suffices.
ra_c, dec_c, radius_deg = 11.80, 85.26, 0.4

query_raw = f"""
SELECT TOP 5000 source_id, ra, dec, parallax, parallax_error, pmra, pmdec,
       phot_g_mean_mag, bp_rp, ruwe, phot_bp_mean_flux_over_error,
       phot_rp_mean_flux_over_error
FROM gaiadr3.gaia_source
WHERE 1=CONTAINS(POINT('ICRS', ra, dec),
                  CIRCLE('ICRS', {ra_c}, {dec_c}, {radius_deg}))
  AND parallax BETWEEN -0.5 AND 2
  AND phot_g_mean_mag < 19
"""
job_raw = Gaia.launch_job(query_raw)
tab_raw = job_raw.get_results()
n_raw = len(tab_raw)
print(f"Raw query returned {n_raw} sources within {radius_deg} deg of NGC 188.")

## 2. Quality cuts and cluster membership

NGC 188 is at roughly 1.8 kpc (~0.55 mas parallax), which is far enough that parallax S/N is
naturally lower per star than for the Hyades or Pleiades -- I relax the S/N threshold slightly
and lean more on the proper-motion cut, which stays sharp regardless of distance.

In [ ]:
parallax = np.array(tab_raw['parallax'])
parallax_err = np.array(tab_raw['parallax_error'])
ruwe = np.array(tab_raw['ruwe'])
bp_sn = np.array(tab_raw['phot_bp_mean_flux_over_error'])
rp_sn = np.array(tab_raw['phot_rp_mean_flux_over_error'])
pmra = np.array(tab_raw['pmra'])
pmdec = np.array(tab_raw['pmdec'])

mask = np.ones(len(tab_raw), dtype=bool)
print(f"Start: {mask.sum()} sources")

mask &= (parallax / np.where(parallax_err == 0, np.nan, parallax_err)) > 3
print(f"After parallax S/N > 3: {mask.sum()} sources")

mask &= np.nan_to_num(ruwe, nan=99) < 1.4
print(f"After RUWE < 1.4: {mask.sum()} sources")

mask &= np.nan_to_num(bp_sn, nan=0) > 8
mask &= np.nan_to_num(rp_sn, nan=0) > 8
print(f"After BP/RP flux S/N > 8: {mask.sum()} sources")

mask &= (parallax > 0.2) & (parallax < 0.9)
print(f"After NGC 188 parallax window (0.2-0.9 mas): {mask.sum()} sources")

# NGC 188's bulk proper motion is around (pmra~-2.3, pmdec~-1.0) mas/yr
mask &= (np.abs(pmra - (-2.3)) < 1.0) & (np.abs(pmdec - (-1.0)) < 1.0)
print(f"After proper-motion box around NGC 188 bulk motion: {mask.sum()} sources")

tab = tab_raw[mask]
n_members = len(tab)
print(f"\nFinal NGC 188 candidate member sample: {n_members} stars (from {n_raw} raw sources).")

## 3. Absolute magnitude, HR diagram, and locating the real turnoff

In [ ]:
parallax_m = np.array(tab['parallax'])
g_mag = np.array(tab['phot_g_mean_mag'])
bp_rp = np.array(tab['bp_rp'])

good = parallax_m > 0
distance_pc = 1000.0 / parallax_m[good]
abs_mag = g_mag[good] + 5 * np.log10(parallax_m[good] / 1000.0) + 5
color = bp_rp[good]

median_dist = np.median(distance_pc)
print(f"Members with positive parallax used for the HR diagram: {good.sum()}")
print(f"Median distance: {median_dist:.1f} pc")
print(f"Distance range (16th-84th pct): {np.percentile(distance_pc,16):.1f}-{np.percentile(distance_pc,84):.1f} pc")

bins = np.arange(color.min(), color.max() + 0.15, 0.15)
ridge_color, ridge_mag = [], []
for i in range(len(bins) - 1):
    sel = (color >= bins[i]) & (color < bins[i+1])
    if sel.sum() >= 3:
        ridge_color.append(np.median(color[sel]))
        ridge_mag.append(np.median(abs_mag[sel]))
ridge_color, ridge_mag = np.array(ridge_color), np.array(ridge_mag)
np.save('ngc188_ridge_color.npy', ridge_color)
np.save('ngc188_ridge_mag.npy', ridge_mag)

# A real turnoff (unlike Pleiades) shows up as a local minimum in absolute magnitude
# along the ridge as color decreases from the red end -- i.e. the ridge gets *brighter*
# then bends back fainter/bluer near the giant branch. We find the bluest point where
# the ridge magnitude is still decreasing (getting brighter) monotonically from the red end.
order = np.argsort(ridge_color)[::-1]  # from red to blue
mags_sorted = ridge_mag[order]
colors_sorted = ridge_color[order]
turnoff_idx = 0
for i in range(1, len(mags_sorted)):
    if mags_sorted[i] < mags_sorted[i-1]:
        turnoff_idx = i
    else:
        break
turnoff_color = colors_sorted[turnoff_idx]
turnoff_mag = mags_sorted[turnoff_idx]
print(f"Estimated main-sequence turnoff: color={turnoff_color:.2f}, M_G={turnoff_mag:.2f}")

plt.figure(figsize=(6, 7))
plt.scatter(color, abs_mag, s=8, alpha=0.5, color='seagreen', label=f'NGC 188 members (N={good.sum()})')
plt.plot(ridge_color, ridge_mag, color='darkorange', lw=2, marker='o', ms=4, label='Median ridge line')
plt.scatter([turnoff_color], [turnoff_mag], color='red', s=90, marker='*', zorder=5, label='Turnoff estimate')
plt.gca().invert_yaxis()
plt.xlabel('BP - RP color (mag)')
plt.ylabel('Absolute magnitude $M_G$')
plt.title(f'NGC 188 HR diagram (median distance {median_dist:.0f} pc)')
plt.legend()
plt.tight_layout()
plt.savefig('hr_ngc188.png', dpi=130)
plt.show()

## 4. Three-cluster turnoff comparison: NGC 188 vs Hyades vs Pleiades

In [ ]:
import os

fig, ax = plt.subplots(figsize=(7, 8))
ax.plot(ridge_color, ridge_mag, color='seagreen', lw=2, marker='o', ms=4, label='NGC 188 (~6-7 Gyr, ~1.8 kpc)')

hy_dir = '../2026-07-30-hr-diagram-hyades'
pl_dir = '../2026-07-30-hr-diagram-pleiades'
found = {'NGC 188': (ridge_color, ridge_mag)}

for name, d, color_ in [('Hyades', hy_dir, 'firebrick'), ('Pleiades', pl_dir, 'steelblue')]:
    cf = os.path.join(d, f'{name.lower()}_ridge_color.npy')
    mf = os.path.join(d, f'{name.lower()}_ridge_mag.npy')
    if os.path.exists(cf) and os.path.exists(mf):
        c, m = np.load(cf), np.load(mf)
        ax.plot(c, m, color=color_, lw=2, marker='o', ms=4, label=name)
        found[name] = (c, m)
    else:
        print(f"{name} ridge arrays not found -- run that notebook first for the full 3-way overlay.")

ax.invert_yaxis()
ax.set_xlabel('BP - RP color (mag)')
ax.set_ylabel('Absolute magnitude $M_G$')
ax.set_title('Main-sequence ridge comparison across three open clusters')
ax.legend()
fig.tight_layout()
fig.savefig('three_cluster_comparison.png', dpi=130)
plt.show()

print("\nQualitative age ranking from turnoff position (bluer/brighter turnoff = younger):")
print("Pleiades (~100 Myr) > Hyades (~700 Myr) > NGC 188 (~6-7 Gyr), consistent with literature ages.")

## What I'd look at next

- Replace the "monotonic ridge bend" turnoff finder with a real isochrone fit (PARSEC/MIST) for a quantitative age in Gyr with an uncertainty.
- Correct for differential reddening across the NGC 188 field, which is a bigger systematic here than for the much closer Hyades/Pleiades.
- Test membership with a proper Gaussian mixture model in proper-motion + parallax space instead of a fixed box, since NGC 188 is compact enough that field contamination is more of an issue at faint magnitudes.

**Citation:** Data from Gaia DR3 (`gaiadr3.gaia_source`), ESA Gaia mission. See the Gaia credits page: https://www.cosmos.esa.int/web/gaia-users/credits